# Aprender a decidir: MDP, Bellman y un mundo de casillas

**Facsímil 10 · Aprendizaje por refuerzo** — capítulo 1 (MDP, políticas, retorno y Bellman).

Decidir bajo incertidumbre, con consecuencias que se **encadenan**, es lo que hacen un robot, un sistema
de recomendaciones o un agente que planea pasos. El marco matemático de ese problema es el **proceso de
decisión de Markov** (MDP), y su ecuación fundamental es la de **Bellman**. En este cuaderno resuelves un
pequeño mundo de casillas donde un robot busca la salida evitando una trampa, **sin enseñarle el
camino**: solo le das las recompensas y dejas que Bellman calcule, casilla a casilla, lo que vale estar
en cada sitio. De ahí sale sola la **política óptima**. Es el esqueleto de todo el aprendizaje por
refuerzo.

### Qué vas a aprender
- Qué es un **MDP**: estados, acciones, recompensas, transiciones (con algo de azar) y descuento.
- La **ecuación de Bellman** y cómo iterarla (*value iteration*) hace emerger la política sin programar
  el camino.
- Por qué el valor de una casilla depende del valor de las que vienen **después** (razonar sobre el
  futuro).
- Cómo el **descuento** γ y el **azar** del entorno cambian lo que es prudente hacer.

### Cuánto cuesta
Unos 12 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [1]:
import numpy as np
np.random.seed(0)

# Mundo 3x4. +1 = salida buena, -1 = trampa, # = muro.
FILAS, COLS = 3, 4
META, TRAMPA, MURO = (0,3), (1,3), (1,1)
RECOMPENSA_PASO = -0.04          # moverse cuesta un poquito (incita a no dar rodeos)
GAMMA = 0.9                       # cuanto importa el futuro (descuento)
terminales = {META: 1.0, TRAMPA: -1.0}
ACCIONES = {"^":(-1,0), "v":(1,0), "<":(0,-1), ">":(0,1)}

def estados(): return [(f,c) for f in range(FILAS) for c in range(COLS) if (f,c) != MURO]
def mover(s, a):
    f, c = s[0]+ACCIONES[a][0], s[1]+ACCIONES[a][1]
    return (f,c) if (0<=f<FILAS and 0<=c<COLS and (f,c)!=MURO) else s   # chocar = te quedas
print("Mundo 3x4 con salida (+1), trampa (-1) y un muro. El robot NO sabe el camino.")


Mundo 3x4 con salida (+1), trampa (-1) y un muro. El robot NO sabe el camino.


## 1. Qué es un MDP

Un proceso de decisión de Markov tiene cinco ingredientes:

- **Estados:** las situaciones posibles (aquí, las casillas).
- **Acciones:** lo que el agente puede hacer en cada estado (moverse arriba/abajo/izq/der).
- **Transiciones:** a qué estado lleva cada acción, posiblemente **con azar** (el mundo es resbaladizo:
  no siempre acabas donde querías).
- **Recompensas:** el premio o castigo de cada paso (+1 salir, -1 caer, -0,04 por moverse).
- **Descuento γ:** cuánto valen las recompensas futuras frente a las inmediatas. γ cercano a 1 = el
  agente piensa a largo plazo.

La propiedad de **Markov**: el futuro depende solo del estado actual, no de cómo llegaste a él. Resolver
el MDP es encontrar la **política** (qué hacer en cada estado) que maximiza la recompensa total
esperada a largo plazo. Modelamos el azar del entorno.


In [2]:
def resultados(s, a):
    # 80% la accion elegida, 10% cada perpendicular (mundo resbaladizo)
    perp = {"^":["<",">"], "v":["<",">"], "<":["^","v"], ">":["^","v"]}[a]
    return [(0.8, mover(s,a)), (0.1, mover(s,perp[0])), (0.1, mover(s,perp[1]))]
print("Si el robot intenta ir a la derecha, el 80% de las veces lo logra, pero el 20% resbala.")
print("Ejemplo desde (2,0) intentando '>':", resultados((2,0), ">"))


Si el robot intenta ir a la derecha, el 80% de las veces lo logra, pero el 20% resbala.
Ejemplo desde (2,0) intentando '>': [(0.8, (2, 1)), (0.1, (1, 0)), (0.1, (2, 0))]


## 2. La ecuación de Bellman, iterada

El **valor** de una casilla es la mejor recompensa que puedes esperar desde ahí: lo que ganas al moverte
más el valor (descontado por γ) de donde acabas, promediado sobre el azar. En fórmula:
$$ V(s) = \max_a \sum_{s'} P(s'\mid s,a)\,\big[\,r + \gamma\,V(s')\,\big] $$
No sabemos $V$ de antemano, así que la calculamos **iterando**: empezamos con ceros y aplicamos la
fórmula una y otra vez hasta que los valores dejan de cambiar (*value iteration*). Es un punto fijo que
converge.


In [3]:
V = {s: 0.0 for s in estados()}
for s, r in terminales.items(): V[s] = r

for iteracion in range(1, 200):
    delta = 0
    for s in estados():
        if s in terminales: continue
        mejor = max(sum(p*(RECOMPENSA_PASO + GAMMA*V[s2]) for p, s2 in resultados(s, a)) for a in ACCIONES)
        delta = max(delta, abs(mejor - V[s])); V[s] = mejor
    if delta < 1e-6: break

print(f"Convergio en {iteracion} iteraciones.\n")
print("VALOR de cada casilla (lo que vale estar ahi):")
for f in range(FILAS):
    print("  " + "  ".join(f"{V.get((f,c),0):+.2f}" if (f,c)!=MURO else "  ##  " for c in range(COLS)))


Convergio en 15 iteraciones.

VALOR de cada casilla (lo que vale estar ahi):
  +0.51  +0.65  +0.80  +1.00
  +0.40    ##    +0.49  -1.00
  +0.30  +0.25  +0.34  +0.13


## 3. La política óptima sale sola

Una vez sabes cuánto vale cada casilla, la decisión en cada una es trivial: ir hacia la vecina más
valiosa (en esperanza). Nadie programó el camino; **emergió** de las recompensas y de Bellman. Es la
diferencia entre *decir* el camino y *deducirlo* del valor.


In [4]:
def politica(s):
    return max(ACCIONES, key=lambda a: sum(p*(RECOMPENSA_PASO + GAMMA*V[s2]) for p, s2 in resultados(s, a)))

print("POLITICA optima (que hacer en cada casilla):")
for f in range(FILAS):
    fila = []
    for c in range(COLS):
        s = (f,c)
        if s == MURO: fila.append(" # ")
        elif s == META: fila.append(" +1")
        elif s == TRAMPA: fila.append(" -1")
        else: fila.append(f"  {politica(s)}")
    print("  " + " ".join(fila))
print("\nLas flechas evitan la trampa y rodean el muro. Nadie las programo: salen del valor.")


POLITICA optima (que hacer en cada casilla):
    >   >   >  +1
    ^  #    ^  -1
    ^   >   ^   <

Las flechas evitan la trampa y rodean el muro. Nadie las programo: salen del valor.


**Lo que acaba de pasar.** No le dijimos al robot «ve a la derecha y luego arriba». Le dimos
recompensas y la ecuación de Bellman calculó el valor de cada casilla; la política es solo «ir hacia el
valor». Fíjate en que las flechas **se alejan de la trampa** incluso en casillas cercanas: el valor
negativo de la trampa se **propaga** a los vecinos. Eso es razonar sobre el futuro, no sobre el paso
inmediato.


## 4. Experimento: el descuento γ cambia el carácter del agente

γ controla cuánto le importa al agente el futuro. Con γ alto (cerca de 1), piensa a largo plazo y acepta
rodeos para llegar bien. Con γ bajo, se vuelve cortoplacista: prefiere el premio inmediato y arriesga
más. Lo medimos resolviendo el mundo con varios γ y mirando cuántas casillas cambian de política y
cuánto vale la casilla de inicio.


In [5]:
def resuelve(gamma):
    V = {s: 0.0 for s in estados()}
    for s, r in terminales.items(): V[s] = r
    for _ in range(300):
        d = 0
        for s in estados():
            if s in terminales: continue
            mejor = max(sum(p*(RECOMPENSA_PASO + gamma*V[s2]) for p, s2 in resultados(s, a)) for a in ACCIONES)
            d = max(d, abs(mejor - V[s])); V[s] = mejor
        if d < 1e-7: break
    pol = {s: max(ACCIONES, key=lambda a: sum(p*(RECOMPENSA_PASO + gamma*V[s2]) for p, s2 in resultados(s, a)))
           for s in estados() if s not in terminales}
    return V, pol

print("gamma | valor del inicio (2,0) | politica del inicio")
for g in [0.99, 0.9, 0.5, 0.1]:
    Vg, polg = resuelve(g)
    print(f"  {g:>4} |        {Vg[(2,0)]:+.3f}         |        {polg[(2,0)]}")
print("\nCon gamma alto el inicio vale mas (el futuro premio cuenta); con gamma bajo, el agente apenas")
print("valora llegar a la meta lejana: se vuelve cortoplacista.")


gamma | valor del inicio (2,0) | politica del inicio
  0.99 |        +0.651         |        ^
   0.9 |        +0.296         |        ^
   0.5 |        -0.062         |        ^
   0.1 |        -0.044         |        ^

Con gamma alto el inicio vale mas (el futuro premio cuenta); con gamma bajo, el agente apenas
valora llegar a la meta lejana: se vuelve cortoplacista.


## 5. Pruébalo tú

1. **Sube el coste del paso** a `-0,2`: el robot tendrá tanta prisa que aceptará más riesgo. Las
   recompensas moldean el comportamiento, a veces de formas que no esperas (*reward shaping*).
2. **Haz el mundo menos resbaladizo** (0,95 / 0,025 / 0,025): con menos azar, ¿se atreve a pasar más cerca
   de la trampa? La incertidumbre cambia lo que es prudente.
3. **Mueve la trampa** junto a la meta y observa cómo la política rodea con más cuidado.
4. **Policy iteration:** investiga el método alternativo (evaluar una política y mejorarla, en vez de
   iterar valores). Llega a lo mismo por otro camino.


## 6. Errores comunes

- **Olvidar el azar de las transiciones.** En el mundo real, las acciones no siempre salen como quieres;
  la política óptima tiene eso en cuenta.
- **Poner γ = 1 sin pensar.** Sin descuento, en problemas sin fin los valores pueden no converger. γ < 1
  da convergencia y modela «el futuro vale menos».
- **Diseñar mal las recompensas.** Es el verdadero arte (*reward engineering*, capítulo 6): recompensas
  mal puestas producen comportamientos absurdos pero «óptimos».
- **Confundir valor y recompensa.** La recompensa es inmediata; el valor mira todo el futuro esperado.


## 7. Qué te llevas

- Un **MDP** es estados, acciones, recompensas, un poco de azar y un descuento; resolverlo es encontrar la
  **política** que maximiza el retorno a largo plazo.
- La **ecuación de Bellman** liga el valor de un estado con el de los que le siguen; iterarla hace emerger
  la política sin programar el camino.
- Las **recompensas** son el lenguaje con el que defines lo que quieres, y el **descuento** γ fija el
  horizonte: diseñarlos bien es el verdadero arte (capítulo 6).

**Para seguir:** el siguiente cuaderno ataca el otro gran problema del refuerzo: cuando ni siquiera sabes
las recompensas y tienes que **explorar** para descubrirlas (bandits).


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*